# 🤖 Processo de Decisão de Markov (MDP)
## Demonstração Didática — MBA em IA

---

### O que é um MDP?

Um **Processo de Decisão de Markov** é um framework matemático para modelar decisões sequenciais em ambientes incertos.

Todo MDP é definido por 4 componentes:

| Componente | Símbolo | Descrição |
|---|---|---|
| **Estados** | S | Todas as situações possíveis do ambiente |
| **Ações** | A | O que o agente pode fazer em cada estado |
| **Recompensas** | R | Sinal de feedback do ambiente |
| **Transições** | P(s'\|s,a) | Probabilidade de ir para o estado s' ao executar ação a no estado s |

### O problema: Navegação em um Labirinto

Um agente precisa sair de um ponto e chegar ao **objetivo (★)**.
O agente não controla perfeitamente seus movimentos:
- **80%** de chance de ir para onde planejou
- **20%** de chance de ficar no mesmo lugar

**Objetivo:** encontrar a política ótima — qual ação tomar em cada estado para maximizar a recompensa acumulada.

---
## 1. Importando bibliotecas

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
from IPython.display import clear_output
import time

print('✅ Bibliotecas importadas com sucesso!')

---
## 2. Definindo o Ambiente

O labirinto é uma grade onde cada célula é um **estado**:

- `1` → Parede (estado inválido)
- `0` → Espaço livre (o agente pode estar aqui)
- `9` → Objetivo ★ (recompensa = 1.0)

Vamos visualizar o labirinto antes de qualquer aprendizado:

In [ ]:
# Definição do labirinto
# 1 = Parede | 0 = Livre | 9 = Objetivo
maze = np.array([
    [1, 1, 1, 1, 1, 1],
    [1, 0, 1, 0, 0, 1],
    [1, 0, 1, 0, 1, 1],
    [1, 0, 0, 0, 0, 1],
    [1, 0, 1, 0, 0, 1],
    [1, 0, 0, 1, 9, 1],
    [1, 1, 1, 1, 1, 1]
])

num_rows, num_cols = maze.shape
print(f'Labirinto: {num_rows} linhas x {num_cols} colunas')
print(f'Estados livres: {np.sum(maze == 0)}')
print(f'Paredes: {np.sum(maze == 1)}')
print(f'Objetivo: {np.sum(maze == 9)} célula')

In [ ]:
def plot_maze(V, title='Labirinto', policy=None, show_path=None):
    """
    Visualiza o labirinto com os valores V(s) de cada estado.
    
    Parâmetros:
        V         : matriz de valores dos estados
        title     : título do gráfico
        policy    : dicionário {(row,col): ação} para mostrar as setas
        show_path : lista de (row,col) para destacar o caminho ótimo
    """
    fig, ax = plt.subplots(figsize=(7, 8))
    
    arrows = {'up': '↑', 'down': '↓', 'left': '←', 'right': '→'}
    
    for row in range(num_rows):
        for col in range(num_cols):
            val = maze[row, col]
            
            if val == 1:  # Parede
                color = '#4a4f5c'
                ax.add_patch(plt.Rectangle((col, num_rows - row - 1), 1, 1,
                                           color=color, zorder=1))
            
            elif val == 9:  # Objetivo
                ax.add_patch(plt.Rectangle((col, num_rows - row - 1), 1, 1,
                                           color='#4CAF50', zorder=1))
                ax.text(col + 0.5, num_rows - row - 0.5, '★',
                        ha='center', va='center', fontsize=20,
                        color='white', fontweight='bold', zorder=3)
            
            else:  # Célula livre
                v = V[row, col]
                # Cor: branco (baixo valor) → azul (alto valor)
                intensity = min(max(v, 0), 1)
                r = int(255 * (1 - intensity * 0.87))
                g = int(255 * (1 - intensity * 0.41))
                b = 255
                color = f'#{r:02x}{g:02x}{b:02x}'
                ax.add_patch(plt.Rectangle((col, num_rows - row - 1), 1, 1,
                                           color=color, zorder=1))
                
                # Valor numérico
                txt_color = 'white' if intensity > 0.6 else '#333333'
                ax.text(col + 0.5, num_rows - row - 0.65, f'{v:.3f}',
                        ha='center', va='center', fontsize=9,
                        color=txt_color, fontweight='bold', zorder=3)
                
                # Seta da política ótima
                if policy and (row, col) in policy:
                    arrow = arrows[policy[(row, col)]]
                    ax.text(col + 0.5, num_rows - row - 0.35, arrow,
                            ha='center', va='center', fontsize=16,
                            color=txt_color, alpha=0.8, zorder=3)
            
            # Grade
            ax.add_patch(plt.Rectangle((col, num_rows - row - 1), 1, 1,
                                       fill=False, edgecolor='#cccccc',
                                       linewidth=0.5, zorder=2))
    
    # Caminho ótimo
    if show_path:
        for (r, c) in show_path:
            if maze[r][c] != 9:
                ax.plot(c + 0.5, num_rows - r - 0.5, 'o',
                        color='#FF5722', markersize=10, zorder=4)
    
    ax.set_xlim(0, num_cols)
    ax.set_ylim(0, num_rows)
    ax.set_xticks(range(num_cols))
    ax.set_yticks(range(num_rows))
    ax.set_xticklabels(range(num_cols))
    ax.set_yticklabels(range(num_rows - 1, -1, -1))
    ax.set_xlabel('Coluna', fontsize=11)
    ax.set_ylabel('Linha', fontsize=11)
    ax.set_title(title, fontsize=14, fontweight='bold', pad=12)
    ax.grid(False)
    
    # Legenda
    legend_patches = [
        mpatches.Patch(color='#4a4f5c', label='Parede'),
        mpatches.Patch(color='#f0f0ff', label='Valor baixo'),
        mpatches.Patch(color='#2196F3', label='Valor alto'),
        mpatches.Patch(color='#4CAF50', label='Objetivo ★'),
    ]
    ax.legend(handles=legend_patches, loc='upper right',
              fontsize=9, framealpha=0.9)
    
    plt.tight_layout()
    plt.show()

# Inicializa V com zeros e visualiza o labirinto inicial
V = np.zeros((num_rows, num_cols))
plot_maze(V, title='Estado inicial — V(s) = 0 para todos os estados')

---
## 3. A Equação de Bellman

O coração do MDP é a **Equação de Bellman**, que calcula o valor de um estado:

$$V(s) = \max_a \left[ \gamma \cdot \sum_{s'} P(s'|s,a) \cdot V(s') \right]$$

**No nosso labirinto:**
- `γ` (gamma) = fator de desconto (quanto o agente valoriza o futuro)
- 80% de chance de ir para o estado desejado `s'`
- 20% de chance de ficar no mesmo estado `s`

Simplificando:

$$V(s) = \max_a \left[ \gamma \cdot (0.8 \cdot V(s') + 0.2 \cdot V(s)) \right]$$

O estado objetivo tem recompensa fixa: **V(objetivo) = 1.0**

In [ ]:
# Parâmetros do MDP
gamma = 0.9   # Fator de desconto: quanto o agente valoriza recompensas futuras
               # 0 = agente míope (só quer recompensa imediata)
               # 1 = agente paciente (valoriza igualmente passado e futuro)

p_sucesso = 0.8   # Probabilidade de executar a ação com sucesso
p_falha   = 0.2   # Probabilidade de ficar no mesmo lugar

print(f'γ (gamma)     = {gamma}')
print(f'P(sucesso)    = {p_sucesso}')
print(f'P(falha)      = {p_falha}')
print()
print('Equação de Bellman aplicada:')
print(f'V(s) = max_a [ {gamma} × (0.8 × V(s\'próximo) + 0.2 × V(s_atual)) ]')

In [ ]:
def calculate_value(state, action, V):
    """
    Calcula o valor de Bellman para o par (estado, ação).
    
    Parâmetros:
        state  : (row, col) — posição atual no labirinto
        action : 'up' | 'down' | 'left' | 'right'
        V      : matriz atual de valores
    
    Retorna:
        float: valor esperado da ação nesse estado
    """
    row, col = state
    
    # Estados terminais
    if maze[row, col] == 9:   return 1.0   # Chegou ao objetivo!
    if maze[row, col] == 1:   return 0.0   # Parede — inválido
    
    # Calcula o próximo estado se a ação for bem-sucedida
    next_state = (row, col)  # Padrão: fica no lugar
    if action == 'up':    next_state = (max(row - 1, 0), col)
    elif action == 'down':  next_state = (min(row + 1, num_rows - 1), col)
    elif action == 'left':  next_state = (row, max(col - 1, 0))
    elif action == 'right': next_state = (row, min(col + 1, num_cols - 1))
    
    # Se next_state é parede, o agente fica no lugar
    if maze[next_state[0], next_state[1]] == 1:
        next_state = (row, col)
    
    # Equação de Bellman
    valor = gamma * (p_sucesso * V[next_state[0], next_state[1]] +
                     p_falha   * V[row, col])
    return valor


def get_best_action(state, V):
    """Retorna a melhor ação e seu valor para um estado."""
    actions = ['up', 'down', 'left', 'right']
    best_value  = -float('inf')
    best_action = None
    for action in actions:
        v = calculate_value(state, action, V)
        if v > best_value:
            best_value  = v
            best_action = action
    return best_action, best_value


print('✅ Funções definidas!')
print()

# Exemplo prático: estado (5,1) — célula livre no canto inferior esquerdo
estado_exemplo = (5, 1)
print(f'Exemplo — Estado {estado_exemplo}:')
for a in ['up','down','left','right']:
    v = calculate_value(estado_exemplo, a, V)
    print(f'  Ação "{a:5s}" → valor = {v:.4f}')

---
## 4. Algoritmo de Iteração de Valor

O **Value Iteration** atualiza os valores de todos os estados repetidamente até convergir:

1. Para cada estado `s`:
   - Calcule o valor de cada ação possível
   - Atualize `V(s)` com o máximo
2. Repita até que a mudança máxima (`delta`) seja menor que um limiar

### 4a. Iteração passo a passo (didática)

Execute a célula abaixo várias vezes para ver o aprendizado acontecendo:

In [ ]:
# Execute esta célula repetidamente para ver cada iteração
# (Resete na célula abaixo quando quiser começar do zero)

try:
    iteracao_atual
except NameError:
    iteracao_atual = 0
    V = np.zeros((num_rows, num_cols))

# Calcula uma iteração
delta = 0
new_V = V.copy()
policy = {}

for row in range(num_rows):
    for col in range(num_cols):
        if maze[row, col] != 1:  # Ignora paredes
            v_antigo = V[row, col]
            best_action, best_value = get_best_action((row, col), V)
            new_V[row, col] = best_value
            if maze[row, col] == 0:
                policy[(row, col)] = best_action
            delta = max(delta, abs(v_antigo - best_value))

V = new_V
iteracao_atual += 1

print(f'Iteração {iteracao_atual} | Delta máximo: {delta:.6f}')
print('(Delta próximo de 0 → convergência atingida)')

plot_maze(V, title=f'Iteração {iteracao_atual}  |  Delta = {delta:.4f}', policy=policy)

In [ ]:
# 🔄 Reseta para começar do zero
V = np.zeros((num_rows, num_cols))
iteracao_atual = 0
print('✅ Resetado! Execute a célula acima para reiniciar.')

---
### 4b. Convergência completa com animação

In [ ]:
# Roda o algoritmo completo até convergir, mostrando o progresso
V = np.zeros((num_rows, num_cols))
historico_delta = []
iteracao = 0
threshold = 1e-6

print('Iniciando Value Iteration...')
print('=' * 45)

while True:
    delta = 0
    new_V = V.copy()
    policy = {}
    
    for row in range(num_rows):
        for col in range(num_cols):
            if maze[row, col] != 1:
                v_antigo = V[row, col]
                best_action, best_value = get_best_action((row, col), V)
                new_V[row, col] = best_value
                if maze[row, col] == 0:
                    policy[(row, col)] = best_action
                delta = max(delta, abs(v_antigo - best_value))
    
    V = new_V
    iteracao += 1
    historico_delta.append(delta)
    
    if iteracao <= 5 or iteracao % 10 == 0 or delta < threshold:
        print(f'Iteração {iteracao:3d} | Delta: {delta:.8f}', end='')
        print(' ← CONVERGIU! ✅' if delta < threshold else '')
    
    if delta < threshold:
        break

print('=' * 45)
print(f'\nTotal de iterações até convergir: {iteracao}')
print(f'Delta final: {delta:.2e}')

plot_maze(V,
          title=f'Política Ótima — Convergiu em {iteracao} iterações',
          policy=policy)

---
## 5. Convergência — Visualizando o Aprendizado

In [ ]:
# Gráfico de convergência do delta ao longo das iterações
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Gráfico 1: Delta por iteração (escala linear)
axes[0].plot(historico_delta, color='#2196F3', linewidth=2)
axes[0].fill_between(range(len(historico_delta)), historico_delta,
                     alpha=0.15, color='#2196F3')
axes[0].set_xlabel('Iteração', fontsize=11)
axes[0].set_ylabel('Delta máximo', fontsize=11)
axes[0].set_title('Convergência do Algoritmo (escala linear)', fontsize=12, fontweight='bold')
axes[0].axhline(y=1e-6, color='#FF5722', linestyle='--', label='Limiar (1e-6)', alpha=0.7)
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# Gráfico 2: Delta em escala logarítmica — mostra melhor a queda exponencial
axes[1].semilogy(historico_delta, color='#4CAF50', linewidth=2)
axes[1].set_xlabel('Iteração', fontsize=11)
axes[1].set_ylabel('Delta (escala log)', fontsize=11)
axes[1].set_title('Convergência do Algoritmo (escala log)', fontsize=12, fontweight='bold')
axes[1].axhline(y=1e-6, color='#FF5722', linestyle='--', label='Limiar (1e-6)', alpha=0.7)
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3, which='both')

plt.suptitle('O delta cai exponencialmente → o algoritmo sempre converge', fontsize=11, y=1.02)
plt.tight_layout()
plt.show()

---
## 6. Caminho Ótimo

Seguindo as setas da política ótima a partir do ponto inicial:

In [ ]:
def find_optimal_path(start, V, max_steps=30):
    """
    Simula o agente seguindo a política ótima a partir de 'start'.
    Retorna a lista de estados visitados.
    """
    path = [start]
    current = start
    
    direction_delta = {
        'up':    (-1,  0),
        'down':  ( 1,  0),
        'left':  ( 0, -1),
        'right': ( 0,  1)
    }
    
    for step in range(max_steps):
        row, col = current
        if maze[row, col] == 9:
            print(f'✅ Objetivo alcançado em {step} passos!')
            break
        
        best_action, _ = get_best_action(current, V)
        dr, dc = direction_delta[best_action]
        next_row = min(max(row + dr, 0), num_rows - 1)
        next_col = min(max(col + dc, 0), num_cols - 1)
        
        if maze[next_row, next_col] == 1:
            next_row, next_col = row, col
        
        current = (next_row, next_col)
        path.append(current)
        print(f'Passo {step+1:2d}: {(row, col)} →[{best_action:5s}]→ {current}')
    
    return path


# Ponto de partida: (1,1) — primeiro espaço livre
start = (1, 1)
print(f'Simulando caminho a partir de {start}...')
print('-' * 45)
caminho = find_optimal_path(start, V)
print(f'\nTotal de estados no caminho: {len(caminho)}')

plot_maze(V,
          title='Caminho Ótimo (pontos laranjas) seguindo a política aprendida',
          policy=policy,
          show_path=caminho)

---
## 7. Experimento: O impacto do γ (gamma)

**Gamma** controla o quanto o agente valoriza recompensas futuras:
- `γ = 0.5` → agente míope (prefere recompensas imediatas)
- `γ = 0.9` → agente equilibrado
- `γ = 0.99` → agente paciente (valoriza muito o futuro)

Veja como os valores mudam com diferentes gammas:

In [ ]:
def run_value_iteration(gamma_val, threshold=1e-6):
    """Roda o Value Iteration para um gamma específico e retorna V, policy, n_iter."""
    V_local = np.zeros((num_rows, num_cols))
    n_iter = 0
    
    while True:
        delta = 0
        new_V = V_local.copy()
        for row in range(num_rows):
            for col in range(num_cols):
                if maze[row, col] != 1:
                    v_old = V_local[row, col]
                    best_val = -float('inf')
                    for a in ['up','down','left','right']:
                        r, c = row, col
                        nr, nc = r, c
                        if a == 'up':    nr = max(r-1,0)
                        elif a == 'down':  nr = min(r+1,num_rows-1)
                        elif a == 'left':  nc = max(c-1,0)
                        elif a == 'right': nc = min(c+1,num_cols-1)
                        if maze[nr,nc]==1: nr,nc=r,c
                        if maze[r,c]==9:   val=1.0
                        elif maze[r,c]==1: val=0.0
                        else: val=gamma_val*(0.8*V_local[nr,nc]+0.2*V_local[r,c])
                        best_val = max(best_val, val)
                    new_V[row,col] = best_val
                    delta = max(delta, abs(v_old - best_val))
        V_local = new_V
        n_iter += 1
        if delta < threshold:
            break
    return V_local, n_iter


gammas = [0.5, 0.7, 0.9, 0.99]
fig, axes = plt.subplots(1, len(gammas), figsize=(16, 6))

for ax, g in zip(axes, gammas):
    V_g, n = run_value_iteration(g)
    
    # Plota no subplot
    for row in range(num_rows):
        for col in range(num_cols):
            val = maze[row, col]
            if val == 1:
                ax.add_patch(plt.Rectangle((col, num_rows-row-1), 1, 1, color='#4a4f5c'))
            elif val == 9:
                ax.add_patch(plt.Rectangle((col, num_rows-row-1), 1, 1, color='#4CAF50'))
                ax.text(col+0.5, num_rows-row-0.5, '★', ha='center', va='center',
                        fontsize=16, color='white', fontweight='bold')
            else:
                v = V_g[row, col]
                intensity = min(max(v, 0), 1)
                r2 = int(255*(1-intensity*0.87))
                g2 = int(255*(1-intensity*0.41))
                color = f'#{r2:02x}{g2:02x}ff'
                ax.add_patch(plt.Rectangle((col, num_rows-row-1), 1, 1, color=color))
                txt_c = 'white' if intensity > 0.6 else '#333'
                ax.text(col+0.5, num_rows-row-0.5, f'{v:.2f}',
                        ha='center', va='center', fontsize=8, color=txt_c, fontweight='bold')
            ax.add_patch(plt.Rectangle((col, num_rows-row-1), 1, 1, fill=False,
                                       edgecolor='#ccc', linewidth=0.4))
    
    ax.set_xlim(0, num_cols)
    ax.set_ylim(0, num_rows)
    ax.set_title(f'γ = {g}\n({n} iterações)', fontsize=11, fontweight='bold')
    ax.axis('off')

plt.suptitle('Impacto do fator de desconto γ nos valores dos estados',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('\nObservação: com γ menor, os valores são menores (agente não olha longe)')
print('Com γ maior, os valores se propagam mais longe do objetivo.')

---
## 8. Resumo dos Conceitos

| Conceito | Definição | No nosso exemplo |
|---|---|---|
| **Estado (s)** | Situação atual do agente | Célula (linha, coluna) no labirinto |
| **Ação (a)** | Decisão possível | ↑ ↓ ← → |
| **Recompensa (R)** | Feedback do ambiente | +1 ao chegar no objetivo |
| **Transição P(s'\|s,a)** | Probabilidade de ir para s' | 80% sucesso, 20% fica no lugar |
| **V(s)** | Valor esperado de estar em s | Número em cada célula |
| **Política π** | Regra de decisão | Seta em cada célula |
| **γ (gamma)** | Desconto temporal | 0.9 neste exemplo |
| **Delta** | Mudança máxima em V | Critério de convergência |

### Por que isso importa?

MDPs são a base de:
- **Aprendizado por Reforço** (RL) — quando P(s'|s,a) é desconhecido e precisa ser aprendido
- **Robótica** — navegação autônoma
- **Finanças** — estratégias de trading sequencial
- **Saúde** — tratamentos adaptativos personalizados
- **Jogos** — AlphaGo, OpenAI Five

In [ ]:
# Imprime os valores finais de forma organizada
print('Valores finais V(s) — política convergida:')
print('=' * 50)

arrows = {'up':'↑','down':'↓','left':'←','right':'→'}

for row in range(num_rows):
    linha = ''
    for col in range(num_cols):
        if maze[row, col] == 1:
            linha += '  ██  '
        elif maze[row, col] == 9:
            linha += '  ★   '
        else:
            a, _ = get_best_action((row, col), V)
            linha += f' {V[row,col]:.2f}{arrows[a]} '
    print(linha)

print('=' * 50)
print('\n██ = Parede | ★ = Objetivo | .XX↑ = valor + ação ótima')